# 03 - Build aggregates and sync to Rayfin SQL
POC 31: CineScope. Builds the analytical tables from `raw_titles` and
`raw_credits`, then performs the bulk upsert into the Rayfin-provisioned
SQL database (the Fabric App item created by `npx rayfin up`).

Tables written: Title, Person, Principal, YearStat, GenreYearStat.

Design decisions:
- Aggregation happens here, in the notebook layer, not in the frontend
  and not via GraphQL. Rayfin's generated API is a CRUD API.
- UUID primary keys are generated deterministically (uuid5) from the
  natural keys, so re-running this notebook upserts rather than duplicates.
- Direct SQL (pyodbc) is used for the bulk load, not the GraphQL API.
  Loading 50k+ rows through GraphQL mutations would take hours; the SQL
  path takes minutes. Application reads still go through GraphQL.

**Prerequisite:** run `npx rayfin up` once first, so the database exists.
Connection details come from the Fabric App item (SQL endpoint) - see
SETUP_GUIDE.md Phase 5.


In [ ]:
# Rayfin SQL connection - from the Fabric App item created by `npx rayfin up`
SQL_SERVER = ""    # e.g. xxxx.database.fabric.microsoft.com
SQL_DATABASE = ""  # the Rayfin app database name

VOTE_WEIGHTED = True  # career and year scores weighted by vote counts
MIN_PERSON_TITLES = 1 # keep all referenced people; frontend applies its own floor

assert SQL_SERVER and SQL_DATABASE, "Set SQL_SERVER and SQL_DATABASE first"


In [ ]:
import uuid
import pandas as pd

NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")  # fixed namespace

def stable_id(*parts):
    return str(uuid.uuid5(NS, ":".join(str(p) for p in parts)))

titles = spark.table("raw_titles").toPandas()
credits = spark.table("raw_credits").toPandas()

# Deterministic UUIDs
titles["id"] = titles["tmdbKey"].map(lambda k: stable_id("title", k))
credits["person_uuid"] = credits["personTmdbId"].map(lambda i: stable_id("person", i))
credits["title_uuid"] = credits["titleKey"].map(lambda k: stable_id("title", k))
credits["principal_id"] = credits.apply(
    lambda r: stable_id("principal", r["titleKey"], r["personTmdbId"], r["category"]), axis=1)
credits = credits.drop_duplicates(subset=["principal_id"])

# Drop credits pointing at titles that fell out of scope
credits = credits[credits["title_uuid"].isin(set(titles["id"]))]
print(f"{len(titles):,} titles, {len(credits):,} credits")


In [ ]:
# Person table with precomputed career stats
joined = credits.merge(
    titles[["id", "voteAverage", "voteCount"]],
    left_on="title_uuid", right_on="id", suffixes=("", "_t"))

def person_stats(g):
    votes = g["voteCount"].sum()
    avg = (g["voteAverage"] * g["voteCount"]).sum() / votes if votes else g["voteAverage"].mean()
    return pd.Series({
        "titleCount": g["titleKey"].nunique(),
        "avgRating": round(float(avg), 2),
        "totalVotes": int(votes),
        "dominantRole": "director" if (g["category"] == "director").mean() >= 0.5 else "cast",
    })

stats = joined.groupby("person_uuid").apply(person_stats).reset_index()
meta = credits.sort_values("ordering").drop_duplicates("person_uuid")[
    ["person_uuid", "personTmdbId", "personName", "knownForDepartment", "profilePath"]]
persons = meta.merge(stats, on="person_uuid")
persons = persons[persons["titleCount"] >= MIN_PERSON_TITLES]
print(f"{len(persons):,} people ({(persons['dominantRole']=='director').sum():,} directors)")


In [ ]:
# YearStat and GenreYearStat
def weighted(g):
    votes = g["voteCount"].sum()
    avg = (g["voteAverage"] * g["voteCount"]).sum() / votes if votes else g["voteAverage"].mean()
    return round(float(avg), 2), int(votes)

year_rows = []
for (year, mt), g in titles.groupby(["releaseYear", "mediaType"]):
    avg, votes = weighted(g)
    rt = g.loc[g["mediaType"] == "movie", "runtimeMinutes"].dropna()
    year_rows.append({
        "id": stable_id("yearstat", year, mt), "statKey": f"{year}-{mt}",
        "year": int(year), "mediaType": mt, "titleCount": len(g),
        "avgRating": avg, "avgRuntime": round(float(rt.mean()), 1) if len(rt) else None,
        "totalVotes": votes})

exploded = titles.assign(genre=titles["genres"].str.split(",")).explode("genre")
exploded = exploded[exploded["genre"].astype(bool)]
genre_rows = []
for (genre, year, mt), g in exploded.groupby(["genre", "releaseYear", "mediaType"]):
    avg, votes = weighted(g)
    genre_rows.append({
        "id": stable_id("genrestat", genre, year, mt), "statKey": f"{genre}-{year}-{mt}",
        "genre": genre, "year": int(year), "mediaType": mt,
        "titleCount": len(g), "avgRating": avg, "totalVotes": votes})

print(f"{len(year_rows):,} year stats, {len(genre_rows):,} genre-year stats")


In [ ]:
# Connect to Rayfin SQL with the notebook identity (Entra token)
import struct, pyodbc
from notebookutils import credentials

token = credentials.getToken("https://database.windows.net/.default").encode("utf-16-le")
token_struct = struct.pack(f"<I{len(token)}s", len(token), token)

conn = pyodbc.connect(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};Encrypt=yes",
    attrs_before={1256: token_struct})  # 1256 = SQL_COPT_SS_ACCESS_TOKEN
cur = conn.cursor()
cur.fast_executemany = True
print("Connected")


In [ ]:
# Bulk upsert helper: stage into a temp table, MERGE into the target.
# Full-refresh semantics for child/stat tables (TRUNCATE then insert) -
# consistent with the 6-month TMDB refresh obligation.

def upsert(table, df, key, truncate=False):
    df = df.where(pd.notnull(df), None)
    cols = list(df.columns)
    col_list = ", ".join(f"[{c}]" for c in cols)
    placeholders = ", ".join("?" for _ in cols)
    cur.execute(f"SELECT TOP 0 {col_list} INTO #stage FROM [{table}]")
    cur.executemany(f"INSERT INTO #stage ({col_list}) VALUES ({placeholders})",
                    df[cols].values.tolist())
    if truncate:
        cur.execute(f"DELETE FROM [{table}]")
        cur.execute(f"INSERT INTO [{table}] ({col_list}) SELECT {col_list} FROM #stage")
    else:
        update_set = ", ".join(f"t.[{c}] = s.[{c}]" for c in cols if c != key)
        cur.execute(f"""
            MERGE [{table}] t USING #stage s ON t.[{key}] = s.[{key}]
            WHEN MATCHED THEN UPDATE SET {update_set}
            WHEN NOT MATCHED THEN INSERT ({col_list}) VALUES ({", ".join("s.[" + c + "]" for c in cols)});
        """)
    cur.execute("DROP TABLE #stage")
    conn.commit()
    print(f"{table}: {len(df):,} rows synced")


In [ ]:
# Order matters: parents before children (FK constraints).
# Principal and the stat tables are full-refresh.

title_df = titles[["id", "tmdbKey", "mediaType", "title", "releaseYear", "decade",
                   "runtimeMinutes", "genres", "voteAverage", "voteCount",
                   "popularity", "posterPath", "originalLanguage", "isSeries"]]

person_df = persons.rename(columns={
    "person_uuid": "id", "personTmdbId": "tmdbKey", "personName": "name"})[
    ["id", "tmdbKey", "name", "knownForDepartment", "profilePath",
     "dominantRole", "titleCount", "avgRating", "totalVotes"]]
person_df["tmdbKey"] = person_df["tmdbKey"].astype(str)

principal_df = credits.rename(columns={
    "principal_id": "id", "title_uuid": "title_id", "person_uuid": "person_id"})[
    ["id", "title_id", "person_id", "category", "ordering", "characterName"]]
principal_df = principal_df[principal_df["person_id"].isin(set(person_df["id"]))]

cur.execute("DELETE FROM [Principal]"); conn.commit()  # children out first
upsert("Title", title_df, "id")
upsert("Person", person_df, "id")
upsert("Principal", principal_df, "id", truncate=True)
upsert("YearStat", pd.DataFrame(year_rows), "id", truncate=True)
upsert("GenreYearStat", pd.DataFrame(genre_rows), "id", truncate=True)

for t in ("Title", "Person", "Principal", "YearStat", "GenreYearStat"):
    print(t, cur.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0])
conn.close()


## Done
Open the deployed CineScope app (hosting URL from `npx rayfin up`) -
all four views should now render.

**Note on table names:** Rayfin generates the SQL schema from the entity
decorators. If the generated table or column names differ from the entity
class names (for example pluralisation or a schema prefix), adjust the
`upsert` targets above. Check with:
`SELECT name FROM sys.tables ORDER BY name;`
